# EDA — dataset local de ojos (`data/raw/eyes`)

1. Carga con `EyeDataLoader` (detección automática del formato).
2. Distribución de clases (ojo / no-ojo) y de ojos por imagen.
3. Resoluciones y canales.
4. Ejemplos aleatorios con cajas y landmarks.
5. Imágenes corruptas y duplicadas.

> Si el formato de etiquetas es ambiguo, el cargador se detiene: define `LABEL_FORMAT` abajo.

In [ ]:
import os, sys, hashlib
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from src.config import RAW_DIR, set_global_seed
from src.data.loader_eyes import EyeDataLoader
from src.data.io import list_images, read_image_rgb
from src.data.eye_labels import LANDMARK_NAMES

set_global_seed(42)
DATA_DIR = RAW_DIR / 'eyes'      # cambia a data/synthetic/eyes para probar sin datos reales
LABEL_FORMAT = 'auto'            # auto | csv | coco | yolo | voc | folders
loader = EyeDataLoader(DATA_DIR, label_format=LABEL_FORMAT)
meta = loader.load_metadata()
train_df, val_df, test_df = loader.split(meta)
print(f'Imágenes: {meta.image_path.nunique()} · filas de entrenamiento: {len(meta)}')
meta.head()

## 1. Clases y ojos por imagen

In [ ]:
per_img = meta.groupby('image_path').agg(n_ojos=('n_eyes', 'first')).reset_index()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
meta['eye_prob'].map({1.0: 'ojo', 0.0: 'no ojo'}).value_counts().plot.bar(ax=ax[0], title='Clase por fila')
per_img['n_ojos'].clip(upper=3).value_counts().sort_index().plot.bar(ax=ax[1], title='Ojos por imagen (3 = 3+)')
split_sizes = pd.Series({'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
split_sizes.plot.bar(ax=ax[2], title='Filas por split')
plt.tight_layout()
pd.crosstab(pd.concat([train_df.assign(split='train'), val_df.assign(split='val'), test_df.assign(split='test')])['split'],
            meta['eye_prob'], normalize='index')

## 2. Resoluciones, modo de color y canales

In [ ]:
rows = []
for p in list_images(DATA_DIR):
    try:
        with Image.open(p) as im:
            rows.append({'path': str(p), 'ancho': im.size[0], 'alto': im.size[1], 'modo': im.mode,
                         'canales': len(im.getbands()), 'formato': im.format, 'bytes': p.stat().st_size})
    except Exception as e:
        rows.append({'path': str(p), 'error': repr(e)})
imgs = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
sns.scatterplot(data=imgs, x='ancho', y='alto', s=10, ax=ax[0]); ax[0].set_title('Resolución')
imgs['modo'].value_counts().plot.bar(ax=ax[1], title='Modo de color (L = gris)')
imgs['formato'].value_counts().plot.bar(ax=ax[2], title='Formato')
plt.tight_layout(); imgs.describe(include='all').T

## 3. Cajas y landmarks

In [ ]:
pos = meta[meta.has_box == 1]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(pos['bw'], bins=40, ax=ax[0]); ax[0].set_title('Ancho de la caja / ancho de la imagen')
cov = pd.Series({n: pos[f'lm{i}_m'].mean() for i, n in enumerate(LANDMARK_NAMES)})
cov.plot.barh(ax=ax[1], xlim=(0, 1), title='Cobertura de landmarks')
plt.tight_layout()

## 4. Ejemplos aleatorios (tal como los ve el modelo)

In [ ]:
ds = loader.build_tfdataset(meta.sample(min(9, len(meta)), random_state=42), cache=False)
x, y, _ = next(iter(ds))
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, img, b, lm, c in zip(axes.ravel(), x.numpy(), y['bbox'].numpy(), y['landmarks'].numpy(), y['eye_prob'].numpy()):
    h, w = img.shape[:2]; ax.imshow(img); ax.axis('off'); ax.set_title('ojo' if c[0] else 'no ojo')
    if c[0]:
        ax.add_patch(plt.Rectangle((b[0]*w, b[1]*h), b[2]*w, b[3]*h, fill=False, ec='lime'))
        pts, m = lm[:12].reshape(-1, 2) * [w, h], lm[12:] > 0.5
        ax.scatter(pts[m, 0], pts[m, 1], s=10, c='red')
plt.tight_layout()

## 5. Imágenes corruptas y duplicadas

* **Corruptas**: Pillow no puede abrirlas o `verify()` falla.
* **Duplicados exactos**: mismo hash MD5.
* **Casi duplicados**: mismo *dHash* de 64 bits (misma imagen recomprimida o redimensionada). Si caen en splits distintos hay fuga de información.

In [ ]:
def dhash(path, size=8):
    with Image.open(path) as im:
        g = np.asarray(im.convert('L').resize((size + 1, size)), dtype=np.int16)
    return ''.join('1' if v else '0' for v in (g[:, 1:] > g[:, :-1]).ravel())

corrupt, md5s, dh = [], {}, {}
for p in list_images(DATA_DIR):
    try:
        with Image.open(p) as im:
            im.verify()
        md5s[str(p)] = hashlib.md5(p.read_bytes()).hexdigest()
        dh[str(p)] = dhash(p)
    except Exception as e:
        corrupt.append((str(p), repr(e)))
h = pd.DataFrame({'md5': pd.Series(md5s), 'dhash': pd.Series(dh)})
exact = h[h.duplicated('md5', keep=False)].sort_values('md5')
near = h[h.duplicated('dhash', keep=False) & ~h.index.isin(exact.index)].sort_values('dhash')
split_of = {p: s for s, d in (('train', train_df), ('val', val_df), ('test', test_df)) for p in d.image_path}
leak = h[h.duplicated('dhash', keep=False)].assign(split=lambda d: d.index.map(split_of)).groupby('dhash')['split'].nunique()
print(f'Corruptas: {len(corrupt)} · duplicados exactos: {len(exact)} · casi duplicados: {len(near)} · grupos repartidos en varios splits: {(leak > 1).sum()}')
corrupt[:10]

## Conclusiones

Anotar: balance ojo/no-ojo, resoluciones atípicas, imágenes en gris, landmarks faltantes, corruptas a eliminar y duplicados que haya que agrupar antes de dividir.